# 12-2 Model Selection for Logistic Regression

**Course:** Models of Statistical Analysis — Universidad de los Andes  
**Professor:** Alejandra Tabares

---

## Introduction

Fitting a logistic regression model often involves choosing among several candidate models that differ in the set of predictor variables included. **Model selection** aims to identify the model that best balances goodness-of-fit against complexity, avoiding both underfitting and overfitting.

The main strategies we cover this week are:

1. **Information criteria** (AIC and BIC): penalise model complexity within a maximum-likelihood framework.
2. **Stepwise selection**: automated variable selection driven by AIC.
3. **Cross-validation**: estimate out-of-sample predictive performance directly from the data.
4. **Likelihood Ratio Test (LRT)**: formal hypothesis test for comparing nested models.

All examples are self-contained: data are simulated in code.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, validation_curve
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

np.random.seed(0)
print("Libraries loaded.")

## 1. Simulating Data

We simulate a dataset with 500 observations and 6 predictors. Only 3 of the 6 predictors are truly associated with the outcome; the other 3 are noise variables. This mimics a realistic model selection scenario.

In [ ]:
# ── Simulate binary outcome data ──────────────────────────────────────────────
n = 500

# True signal predictors
x1 = np.random.normal(0, 1, n)
x2 = np.random.normal(0, 1, n)
x3 = np.random.normal(0, 1, n)

# Noise predictors
x4 = np.random.normal(0, 1, n)
x5 = np.random.normal(0, 1, n)
x6 = np.random.normal(0, 1, n)

# True log-odds: only x1, x2, x3 matter
log_odds = -0.5 + 1.2 * x1 - 0.8 * x2 + 0.6 * x3
prob = 1 / (1 + np.exp(-log_odds))
y = np.random.binomial(1, prob, n)

df = pd.DataFrame({"y": y, "x1": x1, "x2": x2, "x3": x3,
                   "x4": x4, "x5": x5, "x6": x6})

print(f"Dataset shape: {df.shape}")
print(f"Class balance — 0: {(y==0).sum()}, 1: {(y==1).sum()}")
df.head()

## 2. AIC and BIC with `statsmodels`

The **Akaike Information Criterion (AIC)** and **Bayesian Information Criterion (BIC)** are derived from the log-likelihood of the fitted model, penalised for the number of estimated parameters $k$:

$$\text{AIC} = -2\,\ell(\hat{\boldsymbol{\beta}}) + 2k$$

$$\text{BIC} = -2\,\ell(\hat{\boldsymbol{\beta}}) + k\,\ln(n)$$

where $\ell(\hat{\boldsymbol{\beta}})$ is the maximised log-likelihood, $k$ is the number of parameters (including the intercept), and $n$ is the sample size.

**Lower values are better.** BIC applies a heavier penalty for larger samples and tends to select more parsimonious models than AIC.

Both criteria are available directly from a `statsmodels` fitted logit object.

In [ ]:
# ── Fit three nested models with statsmodels ──────────────────────────────────
formula_null = "y ~ 1"
formula_true = "y ~ x1 + x2 + x3"
formula_full = "y ~ x1 + x2 + x3 + x4 + x5 + x6"

fit_null = smf.logit(formula_null, data=df).fit(disp=False)
fit_true = smf.logit(formula_true, data=df).fit(disp=False)
fit_full = smf.logit(formula_full, data=df).fit(disp=False)

comparison = pd.DataFrame({
    "Model"     : ["Null (intercept only)", "True (x1+x2+x3)", "Full (all 6 vars)"],
    "k"         : [1, 4, 7],
    "Log-Lik"   : [fit_null.llf, fit_true.llf, fit_full.llf],
    "AIC"       : [fit_null.aic, fit_true.aic, fit_full.aic],
    "BIC"       : [fit_null.bic, fit_true.bic, fit_full.bic],
})

print(comparison.round(2).to_string(index=False))

In [ ]:
# ── Visualise AIC and BIC across models ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
models = ["Null", "True (x1–x3)", "Full (x1–x6)"]

for ax, metric in zip(axes, ["AIC", "BIC"]):
    values = comparison[metric].values
    colors = ["steelblue" if v == values.min() else "lightsteelblue" for v in values]
    bars = ax.bar(models, values, color=colors, edgecolor="black", width=0.5)
    ax.set_title(metric, fontsize=13)
    ax.set_ylabel("Value", fontsize=11)
    ax.set_ylim([values.min() * 0.97, values.max() * 1.02])
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f"{val:.1f}", ha="center", va="bottom", fontsize=9)

fig.suptitle("AIC and BIC by Model (lower is better)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3. Forward Stepwise Selection Using AIC

Stepwise selection is a greedy search strategy that adds (forward) or removes (backward) predictors one at a time based on a criterion — typically AIC. `statsmodels` does not have a built-in stepwise function, so we implement **forward selection** manually.

**Algorithm (forward selection):**
1. Start with the null model (intercept only).
2. For each candidate variable not yet in the model, fit the model with that variable added.
3. Select the variable whose addition yields the lowest AIC.
4. If the new AIC is lower than the current model's AIC, add the variable and repeat from step 2.
5. Stop when no variable addition decreases the AIC.

In [ ]:
# ── Forward stepwise selection by AIC ────────────────────────────────────────
def forward_selection_aic(data, response, candidates):
    """
    Forward selection for logistic regression using AIC.
    Returns a DataFrame logging each step.
    """
    selected    = []
    remaining   = list(candidates)
    current_aic = smf.logit(f"{response} ~ 1", data=data).fit(disp=False).aic
    log         = [{"Step": 0, "Variable Added": "(intercept only)", "AIC": current_aic}]

    while remaining:
        best_aic = current_aic
        best_var = None

        for var in remaining:
            formula = f"{response} ~ {' + '.join(selected + [var])}"
            aic_candidate = smf.logit(formula, data=data).fit(disp=False).aic
            if aic_candidate < best_aic:
                best_aic = aic_candidate
                best_var = var

        if best_var is None:
            break  # No improvement found — stop

        selected.append(best_var)
        remaining.remove(best_var)
        current_aic = best_aic
        log.append({"Step": len(selected), "Variable Added": best_var, "AIC": current_aic})

    return pd.DataFrame(log), selected


candidates = ["x1", "x2", "x3", "x4", "x5", "x6"]
selection_log, final_vars = forward_selection_aic(df, "y", candidates)

print("Forward Selection Log:")
print(selection_log.round(2).to_string(index=False))
print(f"\nSelected variables: {final_vars}")

In [ ]:
# ── Plot AIC progression during forward selection ────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))

ax.plot(selection_log["Step"], selection_log["AIC"],
        marker="o", color="steelblue", lw=2, markersize=8)

for _, row in selection_log.iterrows():
    ax.annotate(row["Variable Added"],
                xy=(row["Step"], row["AIC"]),
                xytext=(5, 6), textcoords="offset points", fontsize=9)

ax.set_xlabel("Step", fontsize=11)
ax.set_ylabel("AIC", fontsize=11)
ax.set_title("AIC Progression — Forward Stepwise Selection", fontsize=13)
ax.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

## 4. Cross-Validation for Logistic Regression

Information criteria are computed on the **training data**, which means they rely on asymptotic approximations and may not directly reflect out-of-sample performance. **Cross-validation** provides a direct, empirical estimate of generalisation error.

### $k$-Fold Cross-Validation

1. Partition the data into $k$ folds of roughly equal size.
2. For each fold $i = 1, \ldots, k$:
   - Use folds $\neq i$ as the training set.
   - Evaluate the fitted model on fold $i$ (the validation set).
3. Average the $k$ validation scores to estimate the generalisation metric.

**Stratified** $k$-fold preserves the class proportion in each fold — recommended for binary outcomes.

Common choices: $k = 5$ or $k = 10$. Larger $k$ reduces bias but increases variance and computation time.

In [ ]:
# ── k-fold cross-validation for the three candidate models ───────────────────
X = df[["x1", "x2", "x3", "x4", "x5", "x6"]].values
X_true = df[["x1", "x2", "x3"]].values
y_arr  = df["y"].values

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)

pipe_true = Pipeline([("scaler", StandardScaler()),
                      ("lr", LogisticRegression(C=1e6, max_iter=500))])
pipe_full = Pipeline([("scaler", StandardScaler()),
                      ("lr", LogisticRegression(C=1e6, max_iter=500))])

cv_auc_true = cross_val_score(pipe_true, X_true, y_arr, cv=cv, scoring="roc_auc")
cv_auc_full = cross_val_score(pipe_full, X,      y_arr, cv=cv, scoring="roc_auc")

print("10-Fold CV Results (AUC):")
print(f"  True model  (x1+x2+x3) : {cv_auc_true.mean():.4f} ± {cv_auc_true.std():.4f}")
print(f"  Full model  (x1–x6)    : {cv_auc_full.mean():.4f} ± {cv_auc_full.std():.4f}")

In [ ]:
# ── Visualise cross-validated AUC distributions (box plot) ───────────────────
fig, ax = plt.subplots(figsize=(6, 4))

data_box = [cv_auc_true, cv_auc_full]
labels   = ["True model\n(x1+x2+x3)", "Full model\n(x1–x6)"]

bp = ax.boxplot(data_box, labels=labels, patch_artist=True, widths=0.4,
                medianprops={"color": "black", "lw": 2})

colors = ["steelblue", "darkorange"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel("AUC", fontsize=11)
ax.set_title("10-Fold CV AUC Distribution by Model", fontsize=13)
ax.grid(True, axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

### Validation Curve: Regularisation Strength

A **validation curve** shows how a model's training and cross-validated performance change as a hyperparameter varies. Below we examine the effect of the regularisation parameter $C$ (inverse of regularisation strength) on the cross-validated AUC.

In [ ]:
# ── Validation curve: regularisation strength C ───────────────────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

pipe_val = Pipeline([("scaler", StandardScaler()),
                     ("lr", LogisticRegression(max_iter=500, random_state=0))])

C_range = np.logspace(-3, 3, 20)

train_scores, val_scores = validation_curve(
    pipe_val, X_true, y_arr,
    param_name="lr__C",
    param_range=C_range,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=2),
    scoring="roc_auc",
    n_jobs=-1,
)

train_mean = train_scores.mean(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(7, 4))

ax.semilogx(C_range, train_mean, label="Training AUC",   color="steelblue",  lw=2)
ax.semilogx(C_range, val_mean,   label="Validation AUC", color="darkorange", lw=2)
ax.fill_between(C_range, val_mean - val_std, val_mean + val_std,
                alpha=0.15, color="darkorange")

ax.axvline(C_range[val_mean.argmax()], color="gray", lw=1.2,
           linestyle="--", label=f"Best C ≈ {C_range[val_mean.argmax()]:.3f}")

ax.set_xlabel("Regularisation parameter C  (log scale)", fontsize=11)
ax.set_ylabel("AUC", fontsize=11)
ax.set_title("Validation Curve — Regularisation Strength", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, which="both", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

best_C = C_range[val_mean.argmax()]
print(f"Best C by validation AUC: {best_C:.4f}")

## 5. Likelihood Ratio Test (LRT) for Nested Models

When one model is a **special case** of another (i.e., they are *nested*), the **Likelihood Ratio Test** provides a formal hypothesis test.

**Setup:** Let Model 0 (restricted) be nested within Model 1 (unrestricted), with $k_0 < k_1$ parameters.

$$H_0: \text{the additional parameters in Model 1 are all zero}$$
$$H_1: \text{at least one additional parameter is non-zero}$$

**Test statistic:**

$$LR = -2\left[\,\ell(\hat{\boldsymbol{\beta}}_0) - \ell(\hat{\boldsymbol{\beta}}_1)\,\right] = 2\left[\,\ell(\hat{\boldsymbol{\beta}}_1) - \ell(\hat{\boldsymbol{\beta}}_0)\,\right]$$

Under $H_0$, $LR \sim \chi^2(k_1 - k_0)$ asymptotically.

A small $p$-value provides evidence that the additional predictors in Model 1 contribute significantly to the fit.

> **Note:** The LRT is only valid for **nested** models fitted on the **same dataset** with the **same estimation method** (maximum likelihood). It cannot be used to compare non-nested models.

In [ ]:
# ── LRT: Null model vs. True model (x1 + x2 + x3) ───────────────────────────
def likelihood_ratio_test(fit_restricted, fit_unrestricted):
    """
    Perform a likelihood ratio test comparing two nested statsmodels logit fits.
    Returns the LR statistic, degrees of freedom, and p-value.
    """
    lr_stat = 2 * (fit_unrestricted.llf - fit_restricted.llf)
    df_diff = fit_unrestricted.df_model - fit_restricted.df_model
    p_value = stats.chi2.sf(lr_stat, df=df_diff)   # survival function = 1 - CDF
    return lr_stat, int(df_diff), p_value


# ── Test 1: Null  vs. True (adding x1, x2, x3) ──────────────────────────────
lr1, df1, p1 = likelihood_ratio_test(fit_null, fit_true)
print("LRT 1 — Null model vs. True model (x1+x2+x3):")
print(f"  LR statistic = {lr1:.4f}")
print(f"  Degrees of freedom = {df1}")
print(f"  p-value = {p1:.6f}\n")

# ── Test 2: True  vs. Full (adding x4, x5, x6) ──────────────────────────────
lr2, df2, p2 = likelihood_ratio_test(fit_true, fit_full)
print("LRT 2 — True model (x1+x2+x3) vs. Full model (x1–x6):")
print(f"  LR statistic = {lr2:.4f}")
print(f"  Degrees of freedom = {df2}")
print(f"  p-value = {p2:.6f}")

In [ ]:
# ── Visualise LR statistics with chi-squared critical values ──────────────────
alpha = 0.05
cv1 = stats.chi2.ppf(1 - alpha, df=df1)   # critical value for test 1
cv2 = stats.chi2.ppf(1 - alpha, df=df2)   # critical value for test 2

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, lr_stat, df_chi, cv, title in zip(
    axes,
    [lr1, lr2],
    [df1, df2],
    [cv1, cv2],
    ["LRT 1: Null vs. True\n(df = 3)",
     "LRT 2: True vs. Full\n(df = 3)"],
):
    x_max = max(lr_stat, cv) * 1.5
    x = np.linspace(0.01, x_max, 300)
    y_chi = stats.chi2.pdf(x, df=df_chi)

    ax.plot(x, y_chi, lw=2, color="steelblue", label=f"χ²({df_chi})")

    # Rejection region
    x_rej = x[x >= cv]
    ax.fill_between(x_rej, stats.chi2.pdf(x_rej, df=df_chi),
                    alpha=0.25, color="red", label=f"Rejection region (α=0.05)")

    ax.axvline(cv,      color="red",       lw=1.5, linestyle="--", label=f"Critical value = {cv:.2f}")
    ax.axvline(lr_stat, color="darkorange", lw=2,   linestyle="-",  label=f"LR statistic = {lr_stat:.2f}")

    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Test Statistic", fontsize=10)
    ax.set_ylabel("Density", fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, linestyle="--", alpha=0.3)

fig.suptitle("Likelihood Ratio Tests — Chi-Squared Distributions", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Summary of all model selection results ────────────────────────────────────
cv_null = cross_val_score(
    Pipeline([("scaler", StandardScaler()),
              ("lr", LogisticRegression(C=1e6, max_iter=500))]),
    df[["x1"]].values[:, :0],   # empty feature matrix → intercept only via const trick
    y_arr,
    cv=StratifiedKFold(5, shuffle=True, random_state=3),
    scoring="roc_auc",
)

# Re-use previously computed cv_auc_true and cv_auc_full
summary = pd.DataFrame({
    "Model"         : ["Null", "True (x1+x2+x3)", "Full (x1–x6)"],
    "Parameters (k)": [1, 4, 7],
    "Log-Likelihood": [round(fit_null.llf, 2), round(fit_true.llf, 2), round(fit_full.llf, 2)],
    "AIC"           : [round(fit_null.aic, 2),  round(fit_true.aic, 2),  round(fit_full.aic, 2)],
    "BIC"           : [round(fit_null.bic, 2),  round(fit_true.bic, 2),  round(fit_full.bic, 2)],
    "CV-AUC (mean)" : ["—",
                       f"{cv_auc_true.mean():.4f}",
                       f"{cv_auc_full.mean():.4f}"],
})

print("Model Selection Summary:")
print(summary.to_string(index=False))

## Key Takeaways

1. **AIC and BIC** are quick, closed-form criteria based on the log-likelihood. BIC penalises complexity more heavily for large $n$ and tends to select more parsimonious models. Both are computed on the training data and rely on asymptotic theory.

2. **Forward stepwise selection** by AIC is a practical heuristic for automated variable screening, but it does not guarantee finding the globally optimal subset and should be interpreted cautiously.

3. **Cross-validation** provides an empirical estimate of out-of-sample performance without relying on asymptotic approximations. Stratified $k$-fold is the standard choice for binary outcomes.

4. The **validation curve** reveals the bias–variance trade-off for a hyperparameter: small $C$ (strong regularisation) may underfit; large $C$ (weak regularisation) may overfit.

5. The **Likelihood Ratio Test** offers a formal hypothesis test for comparing nested models. A significant result (small $p$-value) means the additional predictors improve fit beyond what would be expected by chance. It cannot be used for non-nested models.